In [1]:
import os
import pickle
import numpy as np
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint

print("✅ All Base Dependencies & AI Libraries Loaded Successfully.")

✅ All Base Dependencies & AI Libraries Loaded Successfully.


In [2]:
import os

BASE_DIR = 'dataset' 
WORKING_DIR = '.' 

vision_base_model = VGG16()

vision_extractor = Model(inputs=vision_base_model.inputs, outputs=vision_base_model.layers[-2].output)
print("✅ Vision Feature Extractor (VGG16) Initialized.")

image_features_dict = {}
image_directory = os.path.join(BASE_DIR, 'Images')

print("⏳ Extracting Deep Visual Features from Image Dataset...")
for image_filename in tqdm(os.listdir(image_directory)):
    img_path = os.path.join(image_directory, image_filename)
    raw_image = load_img(img_path, target_size=(224, 224))
    img_tensor = img_to_array(raw_image)
    img_tensor = img_tensor.reshape((1, img_tensor.shape[0], img_tensor.shape[1], img_tensor.shape[2]))
    img_tensor = preprocess_input(img_tensor)

    extracted_feature = vision_extractor.predict(img_tensor, verbose=0)
    image_id = image_filename.split('.')[0]
    image_features_dict[image_id] = extracted_feature

with open(os.path.join(WORKING_DIR, 'vision_features.pkl'), 'wb') as file:
    pickle.dump(image_features_dict, file)
print("✅ Visual Features Extracted and Saved.")

✅ Vision Feature Extractor (VGG16) Initialized.
⏳ Extracting Deep Visual Features from Image Dataset...


  0%|          | 0/8091 [00:00<?, ?it/s]

✅ Visual Features Extracted and Saved.


In [3]:
caption_filepath = os.path.join(BASE_DIR, 'captions.txt')
with open(caption_filepath, 'r') as file:
    raw_text_data = file.read()

caption_dataset_map = {}
text_lines = raw_text_data.split('\n')

for line in tqdm(text_lines[1:]):
    line_tokens = line.split(',')
    if len(line_tokens) < 2:
        continue
    img_id, img_caption = line_tokens[0], line_tokens[1:]
    img_id = img_id.split('.')[0]
    img_caption = " ".join(img_caption)
    
    if img_id not in caption_dataset_map:
        caption_dataset_map[img_id] = []
    caption_dataset_map[img_id].append(img_caption)

def normalize_text_corpus(dataset_map):
    for key, captions_list in dataset_map.items():
        for i in range(len(captions_list)):
            caption = captions_list[i]
            # Lowercase, remove special chars, and add sequence tags
            caption = caption.lower()
            caption = caption.replace('[^a-z]', '')
            caption = caption.replace('\s+', ' ')
            caption = 'startseq ' + " ".join([word for word in caption.split() if len(word)>1]) + ' endseq'
            captions_list[i] = caption

normalize_text_corpus(caption_dataset_map)
print("✅ NLP Text Normalization Complete.")

  0%|          | 0/40456 [00:00<?, ?it/s]

✅ NLP Text Normalization Complete.


In [8]:
global_vocabulary_list = []
for key in caption_dataset_map:
    for caption in caption_dataset_map[key]:
        global_vocabulary_list.append(caption)

text_tokenizer = Tokenizer()
text_tokenizer.fit_on_texts(global_vocabulary_list)

with open(os.path.join(WORKING_DIR, 'tokenizer.pkl'), 'wb') as file:
    pickle.dump(text_tokenizer, file)

vocab_complexity = len(text_tokenizer.word_index) + 1
max_sequence_length = max(len(caption.split()) for caption in global_vocabulary_list)

print(f"📊 Vocabulary Size: {vocab_complexity} unique words")
print(f"📏 Maximum Caption Sequence Length: {max_sequence_length} words")

📊 Vocabulary Size: 8485 unique words
📏 Maximum Caption Sequence Length: 35 words


In [9]:
dataset_keys = list(caption_dataset_map.keys())
train_split_keys = dataset_keys[:int(len(dataset_keys)*0.90)]
test_split_keys = dataset_keys[int(len(dataset_keys)*0.90):]

def domain_adaptive_data_generator(data_keys, dataset_map, feature_dict, tokenizer, max_len, vocab_sz, batch_sz):
    """
    Yields batches based on exact number of SEQUENCES (not images).
    This prevents RAM overflow and system hangs during Model.fit()
    """
    img_feature_batch, text_seq_batch, target_word_batch = list(), list(), list()
    
    while True:
        for key in data_keys:
            captions = dataset_map[key]
            current_vision_feature = feature_dict[key][0]
            
            for caption in captions:
                encoded_sequence = tokenizer.texts_to_sequences([caption])[0]
                
                for i in range(1, len(encoded_sequence)):
                    in_seq, out_seq = encoded_sequence[:i], encoded_sequence[i]
                    in_seq = pad_sequences([in_seq], maxlen=max_len)[0]
                    out_seq = to_categorical([out_seq], num_classes=vocab_sz)[0]
                    
                    img_feature_batch.append(current_vision_feature)
                    text_seq_batch.append(in_seq)
                    target_word_batch.append(out_seq)

                    if len(img_feature_batch) == batch_sz:
                        yield (np.array(img_feature_batch), np.array(text_seq_batch)), np.array(target_word_batch)
                        img_feature_batch, text_seq_batch, target_word_batch = list(), list(), list()

print("✅ Advanced Data Generator Ready (Mac RAM Optimized).")

✅ Advanced Data Generator Ready (Mac RAM Optimized).


In [10]:
# 1. Vision Processing Branch
vision_input_layer = Input(shape=(4096,), name="Image_Features_Input")
vision_dropout = Dropout(0.4, name="Vision_Dropout")(vision_input_layer)
vision_dense = Dense(256, activation='relu', name="Vision_Dense")(vision_dropout)

# 2. NLP Processing Branch
nlp_input_layer = Input(shape=(max_sequence_length,), name="Text_Sequence_Input")
nlp_embedding = Embedding(vocab_complexity, 256, mask_zero=True, name="Text_Embedding")(nlp_input_layer)
nlp_dropout = Dropout(0.4, name="NLP_Dropout")(nlp_embedding)
nlp_lstm = LSTM(256, name="Text_LSTM")(nlp_dropout)

# 3. Context Merger (Decoder)
context_fusion = add([vision_dense, nlp_lstm], name="Decoder_Merge")
decoder_dense = Dense(256, activation='relu', name="Decoder_Dense")(context_fusion)
output_layer = Dense(vocab_complexity, activation='softmax', name="Probability_Output")(decoder_dense)

# Compile Final Architecture
multimodal_model = Model(inputs=[vision_input_layer, nlp_input_layer], outputs=output_layer, name="Image_Caption_Generator_Model")
multimodal_model.compile(loss='categorical_crossentropy', optimizer='adam')

print("🧠 Custom Multimodal Architecture Constructed!")
multimodal_model.summary()

🧠 Custom Multimodal Architecture Constructed!


Model: "Image_Caption_Generator_Model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ Text_Sequence_Input │ (None, 35)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Image_Features_Inp… │ (None, 4096)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Text_Embedding      │ (None, 35, 256)   │  2,172,160 │ Text_Sequence_In… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Vision_Dropout      │ (None, 4096)      │          0 │ Image_Features_I… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ NLP_Dropout         │ (None, 35, 256)   │          0 │ Text_Embedding[0… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 35)        │          0 │ Text_Sequence_In… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Vision_Dense        │ (None, 256)       │  1,048,832 │ Vision_Dropout[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Text_LSTM (LSTM)    │ (None, 256)       │    525,312 │ NLP_Dropout[0][0… │
│                     │                   │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Decoder_Merge (Add) │ (None, 256)       │          0 │ Vision_Dense[0][… │
│                     │                   │            │ Text_LSTM[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Decoder_Dense       │ (None, 256)       │     65,792 │ Decoder_Merge[0]… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Probability_Output  │ (None, 8485)      │  2,180,645 │ Decoder_Dense[0]… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,992,741 (22.86 MB)

 Trainable params: 5,992,741 (22.86 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
import os 
from tensorflow.keras.callbacks import EarlyStopping

epochs = 5 
batch_size = 128

steps_per_epoch = (len(train_split_keys) * 5 * 10) // batch_size
steps_per_epoch = int(steps_per_epoch * 0.5) 

training_generator = domain_adaptive_data_generator(
    train_split_keys, caption_dataset_map, image_features_dict, 
    text_tokenizer, max_sequence_length, vocab_complexity, batch_size
)

print(f"🚀 Initializing Training for {epochs} Epochs...")

early_stop = EarlyStopping(monitor='loss', patience=2, restore_best_weights=True)

multimodal_model.fit(
    training_generator,
    epochs=epochs,
    steps_per_epoch=steps_per_epoch,
    callbacks=[early_stop],
    verbose=1
)

multimodal_model.save(os.path.join(WORKING_DIR, 'image_captioning_model.h5'))
print("Training Complete. Model weights saved successfully.")

🚀 Initializing Training for 5 Epochs...
Epoch 1/5
1422/1422 ━━━━━━━━━━━━━━━━━━━━ 134s 94ms/step - loss: 4.8780
Epoch 2/5
1422/1422 ━━━━━━━━━━━━━━━━━━━━ 143s 101ms/step - loss: 4.2726
Epoch 3/5
1422/1422 ━━━━━━━━━━━━━━━━━━━━ 144s 101ms/step - loss: 3.7894
Epoch 4/5
1422/1422 ━━━━━━━━━━━━━━━━━━━━ 145s 102ms/step - loss: 3.6966
Epoch 5/5
1422/1422 ━━━━━━━━━━━━━━━━━━━━ 149s 105ms/step - loss: 3.4930


Training Complete. Model weights saved successfully.


In [12]:
from nltk.translate.bleu_score import corpus_bleu

def infer_caption(model, visual_features, tokenizer, max_len):
    input_text = 'startseq'
    for _ in range(max_len):
        sequence = tokenizer.texts_to_sequences([input_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_len)
        prediction = model.predict([visual_features, sequence], verbose=0)
        prediction_idx = np.argmax(prediction)
        
        predicted_word = None
        for word, index in tokenizer.word_index.items():
            if index == prediction_idx:
                predicted_word = word
                break
                
        if predicted_word is None:
            break
            
        input_text += " " + predicted_word
        if predicted_word == 'endseq':
            break
            
    return input_text

def evaluate_model_performance(model, captions_map, features_dict, tokenizer, max_len, test_keys):
    ground_truths, model_predictions = list(), list()
    
    print("📊 Calculating BLEU Scores for System Accuracy...")
    for key in tqdm(test_keys):
        true_captions = captions_map[key]
        predicted_caption = infer_caption(model, features_dict[key].reshape((1, 4096)), tokenizer, max_len)
        
        ground_truths.append([caption.split() for caption in true_captions])
        model_predictions.append(predicted_caption.split())
        
    print(f"🎯 BLEU-1 Metric Score: {corpus_bleu(ground_truths, model_predictions, weights=(1.0, 0, 0, 0)):.4f}")
    print(f"🎯 BLEU-2 Metric Score: {corpus_bleu(ground_truths, model_predictions, weights=(0.5, 0.5, 0, 0)):.4f}")

evaluate_model_performance(multimodal_model, caption_dataset_map, image_features_dict, text_tokenizer, max_sequence_length, test_split_keys)

📊 Calculating BLEU Scores for System Accuracy...


  0%|          | 0/810 [00:00<?, ?it/s]

🎯 BLEU-1 Metric Score: 0.5192
🎯 BLEU-2 Metric Score: 0.2879
